# Build Your Own Search Engine

Notes and code for the "Build Your Own Search Engine" workshop.

* Original workshop: https://github.com/alexeygrigorev/build-your-own-search-engine
* Video: https://www.youtube.com/watch?v=nMrGK5QgPVE

We use the Zoomcamp FAQ documents and build a search engine to retrieve them.
The results can later feed a [Q&A RAG system](https://github.com/alexeygrigorev/llm-rag-workshop).

**Outline**

1. Preparing the environment
2. Basics of text search — information retrieval, vector spaces, bag of words, TF-IDF
3. Implementing basic text search — TF-IDF scoring, keyword filtering, a search class
4. Embeddings and vector search — SVD/LSA, NMF, BERT
5. Combining text and vector search
6. Practical implementation aspects — inverted indexes, LSH, Lucene/Elasticsearch, FAISS

## 1. Preparing the environment

Any environment works — Codespaces, a local venv, Colab. We need `requests`,
`pandas`, `scikit-learn` and `jupyter`.

In [2]:
!pip install pandas
!pip install requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 34.9 MB/s  0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 56.3 MB/s  0:00:00s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]━━━━ 1/2 [pandas]

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [1]:
import pandas as pd

### Downloading the data

The FAQ documents come as one JSON file, grouped by course. We flatten it into a
single list of records and copy the course name onto every document, so that later
we can filter by course with one field lookup.

Each record ends up with four fields: `course`, `section`, `question`, `text`.

In [2]:
import requests 

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [3]:
documents[2]

{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
 'section': 'General course-related questions',
 'question': 'Course - Can I still join the course after the start date?',
 'course': 'data-engineering-zoomcamp'}

### Creating the dataframe

The row number in this dataframe is the document id for the rest of the workshop.
Every score array we build stays aligned with it, so `np.argsort` on a score array
gives us row positions we can feed straight back into `df.iloc`.

In [4]:
df = pd.DataFrame(documents, columns=['course', 'section', 'question', 'text'])
df.head()

,course,section,question,text
0,data-engineering-zoomcamp,General course-related questions,Course - When will the course start?,The purpose of this document is to capture fre...
1,data-engineering-zoomcamp,General course-related questions,Course - What are the prerequisites for this c...,GitHub - DataTalksClub data-engineering-zoomca...
2,data-engineering-zoomcamp,General course-related questions,Course - Can I still join the course after the...,"Yes, even if you don't register, you're still ..."
3,data-engineering-zoomcamp,General course-related questions,Course - I have registered for the Data Engine...,You don't need it. You're accepted. You can al...
4,data-engineering-zoomcamp,General course-related questions,Course - What can I do before the course starts?,You can start by installing and setting up all...


In [5]:
df.tail()

,course,section,question,text
943,mlops-zoomcamp,Module 6: Best practices,Github actions: Permission denied error when e...,Problem description\nThis is the step in the c...
944,mlops-zoomcamp,Module 6: Best practices,Managing Multiple Docker Containers with docke...,Problem description\nWhen a docker-compose fil...
945,mlops-zoomcamp,Module 6: Best practices,AWS regions need to match docker-compose,Problem description\nIf you are having problem...
946,mlops-zoomcamp,Module 6: Best practices,Isort Pre-commit,Problem description\nPre-commit command was fa...
947,mlops-zoomcamp,Module 6: Best practices,How to destroy infrastructure created via GitH...,Problem description\nInfrastructure created in...


## 2. Basics of Text Search

### Keyword filtering — the simplest "search"

An exact match on a field. This is a *filter*, not relevance: it answers "which
rows equal this value", not "which rows are most about this topic".

Filtering is still useful — we combine it with relevance scoring later, to restrict
results to a single course.

In [6]:
# Filtering with pandas
df[df.course == "data-engineering-zoomcamp"]

,course,section,question,text
0,data-engineering-zoomcamp,General course-related questions,Course - When will the course start?,The purpose of this document is to capture fre...
1,data-engineering-zoomcamp,General course-related questions,Course - What are the prerequisites for this c...,GitHub - DataTalksClub data-engineering-zoomca...
2,data-engineering-zoomcamp,General course-related questions,Course - Can I still join the course after the...,"Yes, even if you don't register, you're still ..."
3,data-engineering-zoomcamp,General course-related questions,Course - I have registered for the Data Engine...,You don't need it. You're accepted. You can al...
4,data-engineering-zoomcamp,General course-related questions,Course - What can I do before the course starts?,You can start by installing and setting up all...
...,...,...,...,...
430,data-engineering-zoomcamp,Workshop 2 - RisingWave,Unable to Open Dashboard as xdg-open doesn’t o...,Refer to the solution given in the first solut...
431,data-engineering-zoomcamp,Workshop 2 - RisingWave,Resolving Python Interpreter Path Inconsistenc...,Example Error:\nWhen attempting to execute a P...
432,data-engineering-zoomcamp,Workshop 2 - RisingWave,How does windowing work in Sql?,Ans : Windowing in streaming SQL involves defi...
433,data-engineering-zoomcamp,Triggers in Mage via CLI,"Encountering the error ""ModuleNotFoundError: N...","Python 3.12.1, is not compatible with kafka-py..."


Basics of Text Search:
1. Information Retrieval - The process of obtaining relevant information from large datasets based on user queries.
2. Vector Spaces - A mathematical representation where text is converted into vectors (points in space) allowing for quantitative comparison.
3. Bag of Words - A simple text representation model treating each document as a collection of words disregarding grammar and word order but keeping multiplicity.
4. TF-IDF (Term Frequency-Inverse Document Frequency) - A statistical measure used to evaluate how important a word is to a document in a collection or corpus. It increases with the number of times a word appears in the document but is offset by the frequency of the word in the corpus.


### What is Vector Spaces?

- turn the document into vector
- term-document matrix
    - row: documents ('Course - Can I still join the course after the start date?')
    - column: words/tokens
        - join is 1
        - course is 1
        - start is 1
        - date is 1
- in sklearn we have something called CountVectorizer(turning text into vectors)

## 3. Implementing Basic Text Search

### Vectorization

`CountVectorizer` builds the vocabulary from the corpus and turns each document
into a row of word counts. Two knobs matter here:

* `stop_words='english'` drops words like *on*, *not*, *for*, *after* — they occur
  everywhere and carry no signal about what a document is about.
* `min_df=5` keeps only tokens appearing in at least 5 documents. On the full FAQ
  this cuts the vocabulary from thousands of noisy tokens (typos, one-off names)
  down to a manageable set.

In [7]:
!pip3 install -U scikit-learn

  Using cached scikit_learn-1.9.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.9.0-cp312-cp312-macosx_12_0_arm64.whl (8.3 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn]━━━━━ 2/3 [scikit-learn]

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [8]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()
cv

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word n-gram or charactern-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21Since v0.21, if ``input`` is ``filename`` or ``file``, the data isfirst read from the file and then passed to the given callableanalyzer.",'word'


In [9]:
cv.fit(df.text)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word n-gram or charactern-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21Since v0.21, if ``input`` is ``filename`` or ``file``, the data isfirst read from the file and then passed to the given callableanalyzer.",'word'


In [10]:
cv.get_feature_names_out()

array(['00', '00000000e', '0002', ..., '要了解键盘快捷键', '要启用屏幕阅读器支持', '请按ctrl'],
      shape=(6711,), dtype=object)

In [11]:
# There is a lot of noise here, so we will use only 5 documents.
cv = CountVectorizer(min_df=5)
cv.fit(df.text)
cv.get_feature_names_out()

array(['01', '02', '03', ..., 'youtube', 'zip', 'zoomcamp'],
      shape=(1524,), dtype=object)

#### A smaller example

The full corpus produces a matrix too large to read. These five short sentences
give a term-document matrix we can print and inspect by eye.

In [12]:
docs_examples = [
    "Course starts on 15th Jan 2024",
    "Prerequisites listed on GitHub",
    "Submit homeworks after start date",
    "Registration not required for participation",
    "Setup Google Cloud and Python before course"
]

In [13]:
docs_examples

['Course starts on 15th Jan 2024',
 'Prerequisites listed on GitHub',
 'Submit homeworks after start date',
 'Registration not required for participation',
 'Setup Google Cloud and Python before course']

In [14]:
cv = CountVectorizer()
cv.fit(docs_examples)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word n-gram or charactern-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21Since v0.21, if ``input`` is ``filename`` or ``file``, the data isfirst read from the file and then passed to the given callableanalyzer.",'word'


In [15]:
cv.get_feature_names_out() # all the words in the doc_examples

array(['15th', '2024', 'after', 'and', 'before', 'cloud', 'course',
       'date', 'for', 'github', 'google', 'homeworks', 'jan', 'listed',
       'not', 'on', 'participation', 'prerequisites', 'python',
       'registration', 'required', 'setup', 'start', 'starts', 'submit'],
      dtype=object)

In [16]:
X = cv.transform(docs_examples)
X.shape

(5, 25)

In [17]:
X

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 27 stored elements and shape (5, 25)>

In [18]:
X.todense()

matrix([[1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0,
         0, 0, 1, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0,
         0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 1, 0, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1,
         0, 0, 0, 0],
        [0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
         1, 0, 0, 0]])

In [19]:
# If we want to see what is inside
pd.DataFrame(X.todense(), columns=cv.get_feature_names_out())

,15th,2024,after,and,before,cloud,course,date,for,github,...,on,participation,prerequisites,python,registration,required,setup,start,starts,submit
0,1,1,0,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0,1,0
1,0,0,0,0,0,0,0,0,0,1,...,1,0,1,0,0,0,0,0,0,0
2,0,0,1,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,1,0,1
3,0,0,0,0,0,0,0,0,1,0,...,0,1,0,0,1,1,0,0,0,0
4,0,0,0,1,1,1,1,0,0,0,...,0,0,0,1,0,0,1,0,0,0


In [20]:
pd.DataFrame(X.todense(), columns=cv.get_feature_names_out()).T

,0,1,2,3,4
15th,1,0,0,0,0
2024,1,0,0,0,0
after,0,0,1,0,0
and,0,0,0,0,1
before,0,0,0,0,1
cloud,0,0,0,0,1
course,1,0,0,0,1
date,0,0,1,0,0
for,0,0,0,1,0
github,0,1,0,0,0


- turn the document into vector
- term-document matrix
    - row: documents ('Course - Can I still join the course after the start date?')
    - column: words/tokens
        - join is 1
        - course is 1
        - start is 1
        - date is 1
- in sklearn we have something called CountVectorizer(turning text into vectors)
- bag of words
    - order is not important in words.
    - sparse matrix.


#### Removing stop words

Words like *on*, *not*, *for*, *after*, *before* appear in almost every document,
so they add columns without adding any ability to tell documents apart.

In [21]:
# remove stop words like on, not, at, for, after, before
cv = CountVectorizer(stop_words='english')
cv.fit(docs_examples)

,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word n-gram or charactern-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21Since v0.21, if ``input`` is ``filename`` or ``file``, the data isfirst read from the file and then passed to the given callableanalyzer.",'word'


In [22]:
cv.get_feature_names_out()

array(['15th', '2024', 'cloud', 'course', 'date', 'github', 'google',
       'homeworks', 'jan', 'listed', 'participation', 'prerequisites',
       'python', 'registration', 'required', 'setup', 'start', 'starts',
       'submit'], dtype=object)

In [23]:
X = cv.transform(docs_examples)
pd.DataFrame(X.todense(), columns=cv.get_feature_names_out()).T

,0,1,2,3,4
15th,1,0,0,0,0
2024,1,0,0,0,0
cloud,0,0,0,0,1
course,1,0,0,0,1
date,0,0,1,0,0
github,0,1,0,0,0
google,0,0,0,0,1
homeworks,0,0,1,0,0
jan,1,0,0,0,0
listed,0,1,0,0,0


In [24]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words='english', min_df=5)
X = cv.fit_transform(df.text)

names = cv.get_feature_names_out()

df_docs = pd.DataFrame(X.toarray(), columns=names)
df_docs
# more than 900 documents and in the columns we can see the words

,01,02,03,04,05,06,09,10,100,11,...,y_val,yaml,year,yellow,yellow_tripdata_2021,yes,yml,youtube,zip,zoomcamp
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
943,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
944,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
945,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
946,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### TF-IDF (Term Frequency-Inverse Document Frequency) 

A statistical measure used to evaluate how important a word is to a document in a collection or corpus. It increases with the number of times a word appears in the document but is offset by the frequency of the word in the corpus.
Breaking it down:

* **"It increases with the number of times a word appears in the document"** →
  That’s the **TF** part: *Term Frequency*. If a word shows up more often in a document, it gets a higher score.

* **"but is offset by the frequency of the word in the corpus"** →
  That’s the **IDF** part: *Inverse Document Frequency*. If a word appears in **many documents across the whole collection**, it becomes *less special*, so the score is reduced.

Putting it together:

* **Rare words** that appear frequently in a document get **high scores**.
* **Common words** like “yes,” “and,” “the” get **low scores** even if they appear a lot, because they don’t help distinguish documents.
* In our example: `"yes"` appears in many documents → low importance; `"yml"` appears rarely in the corpus → higher importance.



**Setup**

* Corpus size $N=5$ documents.
* Document $D$ has **100 words**.
* In $D$: `"yes"` appears **6** times; `"yml"` appears **2** times.
* Document frequencies: `"yes"` is in **4** of 5 docs; `"yml"` is in **1** of 5 docs.

**Formulas**

* $\text{TF}(t,D) = \frac{\text{count of } t \text{ in } D}{\text{total words in } D}$
* $\text{IDF}(t) = \ln\!\left(\frac{N}{\text{df}(t)}\right)$
* $\text{TF-IDF}(t,D) = \text{TF}(t,D)\times \text{IDF}(t)$

**Numbers**

* TF:

  * $\text{TF}(\text{"yes"},D)=6/100=0.06$
  * $\text{TF}(\text{"yml"},D)=2/100=0.02$
* IDF:

  * $\text{IDF}(\text{"yes"})=\ln(5/4)\approx 0.2231$
  * $\text{IDF}(\text{"yml"})=\ln(5/1)\approx 1.6094$
* TF-IDF:

  * $\text{TF-IDF}(\text{"yes"},D)=0.06\times 0.2231\approx \mathbf{0.0134}$
  * $\text{TF-IDF}(\text{"yml"},D)=0.02\times 1.6094\approx \mathbf{0.0322}$

**Takeaway**
Even though `"yes"` appears more often **in the document**, `"yml"` is much **rarer in the corpus**, so it gets a **\~2.4× higher** TF-IDF score. This is exactly the “more common ⇒ less important” intuition.


### Query-Document Similarity

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer

cv = TfidfVectorizer(stop_words='english', min_df=5)
X = cv.fit_transform(df.text)
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 23808 stored elements and shape (948, 1333)>

The query has to live in the *same* vector space as the documents — same columns,
same order. That is why we call `transform` and not `fit_transform`: the
vectorizer is already fitted on the corpus.

Words in the query that are not in the fitted vocabulary are simply dropped.

In [26]:
query = "Do I need to know python to sign up for the January course?"
q = cv.transform([query])
q.toarray()

array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 1333))

In [27]:
query_dict = dict(zip(names, q.toarray()[0]))
filtered = {k: v for k, v in query_dict.items() if v > 0}
filtered

{'course': np.float64(0.38148200594064524),
 'know': np.float64(0.5608269127690405),
 'need': np.float64(0.29796783250107517),
 'python': np.float64(0.31441356049301333),
 'sign': np.float64(0.5935519664108326)}

In [28]:
doc_dict = dict(zip(names, X.toarray()[2]))
filtered = {k: v for k, v in doc_dict.items() if v > 0}
filtered

{'don': np.float64(0.5310683382058037),
 'final': np.float64(0.38088037206388314),
 'homeworks': np.float64(0.38088037206388314),
 'projects': np.float64(0.2982703425530899),
 'register': np.float64(0.399850779948394),
 'submit': np.float64(0.31724075043760075),
 'yes': np.float64(0.2798913490945963)}

#### Dot Product

#### Dot product

Multiply the query vector and a document vector element-wise, then sum. Only terms
present in *both* contribute — everywhere else one of the two factors is 0. So the
more (important) words they share, the higher the score.

Because it is a dot product, we can score every document at once with a single
matrix multiplication instead of a Python loop. That is what makes this fast.

In [29]:
X.dot(q.T).todense() # cosine similiarity

matrix([[0.19464486],
        [0.        ],
        [0.        ],
        [0.06011641],
        [0.04932915],
        [0.        ],
        [0.        ],
        [0.13477565],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.15899187],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.07431408],
        [0.        ],
        [0.        ],
        [0.05779673],
        [0.07243428],
        [0.        ],
        [0.05174293],
        [0.16373635],
        [0.08076031],
        [0.        ],
        [0.09755254],
        [0.        ],
        [0.21069625],
        [0.12067781],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.06381749],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.00910541],
        [0.02835681],
        [0.05480112],
        [0.        ],
        [0.        ],
        [0.        ],
        [0

#### Cosine similarity

Cosine similarity is the dot product divided by the length of both vectors, so it
measures the *angle* between them and ignores document length — otherwise long
documents would score high just for being long.

`TfidfVectorizer` already returns L2-normalised vectors, so here the dot product
and the cosine similarity give identical results.

In [30]:
from sklearn.metrics.pairwise import cosine_similarity

In [31]:
score = cosine_similarity(X,q).flatten() # same result as the dot product

`np.argsort` sorts ascending and returns *positions*, not values. So the best
matches are at the end — or at the front if we sort `-score` instead.

Note: [np.argpartition](https://numpy.org/doc/stable/reference/generated/numpy.argpartition.html)
does the same job more efficiently, because it only has to find the top k rather
than order the whole array.

In [32]:
import numpy as np
np.argsort(score) # these are index sorted following the score started with zero for example the index
# 524 has zero score, and we're interested with the indexes at the end 27, 806, 577, 445

array([473, 563, 564, 566, 567, 568, 569, 570, 571, 572, 574, 575, 576,
       578, 579, 580, 581, 582, 583, 584, 562, 561, 560, 559, 530, 532,
       533, 534, 535, 536, 537, 538, 542, 585, 544, 548, 549, 550, 551,
       552, 553, 555, 556, 558, 546, 586, 590, 594, 634, 635, 636, 637,
       638, 640, 641, 643, 644, 631, 645, 647, 649, 650, 651, 652, 653,
       654, 655, 657, 646, 528, 630, 627, 595, 597, 600, 601, 602, 604,
       605, 606, 607, 628, 608, 612, 613, 614, 615, 616, 618, 621, 622,
       626, 611, 527, 526, 525, 422, 423, 426, 427, 428, 429, 430, 432,
       437, 421, 441, 443, 444, 447, 453, 460, 461, 462, 463, 466, 442,
       467, 420, 418, 385, 386, 387, 389, 390, 392, 397, 399, 400, 419,
       402, 405, 407, 408, 409, 410, 412, 414, 416, 417, 404, 658, 468,
       472, 499, 501, 504, 505, 506, 507, 508, 509, 510, 498, 512, 514,
       515, 516, 517, 518, 519, 520, 523, 524, 513, 471, 497, 495, 946,
       474, 475, 476, 477, 478, 479, 480, 481, 496, 482, 486, 48

In [33]:
np.argsort(score)[-5:]

array([764,  27, 806, 577, 445])

In [34]:
query = "I just discovered the course, is it too late to join?"
q = cv.transform([query])
q.toarray()

array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 1333))

In [35]:
score = cosine_similarity(X,q).flatten() 

In [36]:
np.argsort(score) [-5:]

array([ 22, 448, 449, 440,   0])

In [37]:
# these indexes represent documents
df.iloc[22].text

"It's up to you which platform and environment you use for the course.\nGithub codespaces or GCP VM are just possible options, but you can do the entire course from your laptop."

In [38]:
df.iloc[449].text

'Yes, you can. You won’t be able to submit some of the homeworks, but you can still take part in the course.\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers’ Projects by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'

In [39]:
df.iloc[0].text

"The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel."

### Vectorizing all the documents

In [40]:
fields = ['section', 'question', 'text']

One vectorizer per field: each field has its own vocabulary and its own corpus
statistics, so a word that is rare in `question` may be common in `text`.

In [41]:
matrices = {}
vectorizers = {}

for f in fields:
    cv = TfidfVectorizer(stop_words='english', min_df=5)
    X = cv.fit_transform(df[f])
    matrices[f] = X
    vectorizers[f] = cv

In [42]:
matrices

{'section': <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 3090 stored elements and shape (948, 66)>,
 'question': <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 3431 stored elements and shape (948, 291)>,
 'text': <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 23808 stored elements and shape (948, 1333)>}

section has 66 tokens, and text has the highest number of tokens which is 1333

In [43]:
vectorizers

{'section': TfidfVectorizer(min_df=5, stop_words='english'),
 'question': TfidfVectorizer(min_df=5, stop_words='english'),
 'text': TfidfVectorizer(min_df=5, stop_words='english')}

### Search


In [44]:
n = len(df)
n

948

In [45]:
score = np.zeros(n)
query = "I just discovered the course, is it too late to join?"

for f in fields:
    q = vectorizers[f].transform([query])
    X = matrices[f]

    f_score = cosine_similarity(X, q).flatten()
    score = score + f_score
    

Filtering by multiplying with a 0/1 mask: documents from another course get a
score of 0, which pushes them to the bottom of the ranking.

In [46]:
filters = {
    'course': 'data-engineering-zoomcamp'
}

In [47]:
for field, value in filters.items():
    mask = (df[field] == value).astype(int).values
    score = score * mask

In [48]:
idx = np.argsort(score)[-11:]

In [49]:
df.iloc[idx]

,course,section,question,text
2,data-engineering-zoomcamp,General course-related questions,Course - Can I still join the course after the...,"Yes, even if you don't register, you're still ..."
11,data-engineering-zoomcamp,General course-related questions,Certificate - Can I follow the course in a sel...,"No, you can only get a certificate if you fini..."
10,data-engineering-zoomcamp,General course-related questions,Course - ​​How many hours per week am I expect...,It depends on your background and previous exp...
3,data-engineering-zoomcamp,General course-related questions,Course - I have registered for the Data Engine...,You don't need it. You're accepted. You can al...
9,data-engineering-zoomcamp,General course-related questions,Course - Which playlist on YouTube should I re...,All the main videos are stored in the Main “DA...
34,data-engineering-zoomcamp,General course-related questions,How can we contribute to the course?,Star the repo! Share it with friends if you fi...
5,data-engineering-zoomcamp,General course-related questions,Course - how many Zoomcamps in a year?,"There are 3 Zoom Camps in a year, as of 2024. ..."
4,data-engineering-zoomcamp,General course-related questions,Course - What can I do before the course starts?,You can start by installing and setting up all...
1,data-engineering-zoomcamp,General course-related questions,Course - What are the prerequisites for this c...,GitHub - DataTalksClub data-engineering-zoomca...
7,data-engineering-zoomcamp,General course-related questions,Course - Can I follow the course after it fini...,"Yes, we will keep all the materials after the ..."


### Search with all the fields & boosting + filtering
We can do it for all the fields. Let's also boost one of the fields - question - to give it more importance than to others

In [50]:
score = np.zeros(n)
query = "I just discovered the course, is it too late to join?"

boosts = {'question': 3}

for f in fields:
    q = vectorizers[f].transform([query])
    X = matrices[f]

    f_score = cosine_similarity(X, q).flatten()
    boost = boosts.get(f, 1.0)
    
    score = score + boost * f_score

In [51]:
filters = {
    'course': 'data-engineering-zoomcamp'
}

In [52]:
for field, value in filters.items():
    mask = (df[field] == value).astype(int).values
    score = score * mask

In [53]:
idx = np.argsort(-score)[:5]

In [54]:
df.iloc[idx]

,course,section,question,text
7,data-engineering-zoomcamp,General course-related questions,Course - Can I follow the course after it fini...,"Yes, we will keep all the materials after the ..."
0,data-engineering-zoomcamp,General course-related questions,Course - When will the course start?,The purpose of this document is to capture fre...
1,data-engineering-zoomcamp,General course-related questions,Course - What are the prerequisites for this c...,GitHub - DataTalksClub data-engineering-zoomca...
4,data-engineering-zoomcamp,General course-related questions,Course - What can I do before the course starts?,You can start by installing and setting up all...
5,data-engineering-zoomcamp,General course-related questions,Course - how many Zoomcamps in a year?,"There are 3 Zoom Camps in a year, as of 2024. ..."


### Putting it all together

In [55]:
class TextSearch:

    def __init__(self, text_fields):
        self.text_fields = text_fields
        self.matrices = {}
        self.vectorizers = {}

    def fit(self, records, vectorizer_params={}):
        self.df = pd.DataFrame(records)

        for f in self.text_fields:
            cv = TfidfVectorizer(**vectorizer_params)
            X = cv.fit_transform(self.df[f])
            self.matrices[f] = X
            self.vectorizers[f] = cv

    def search(self, query, n_results=10, boost={}, filters={}):
        score = np.zeros(len(self.df))

        for f in self.text_fields:
            b = boost.get(f, 1.0)
            q = self.vectorizers[f].transform([query])
            s = cosine_similarity(self.matrices[f], q).flatten()
            score = score + b * s

        for field, value in filters.items():
            mask = (self.df[field] == value).values
            score = score * mask

        idx = np.argsort(-score)[:n_results]
        results = self.df.iloc[idx]
        return results.to_dict(orient='records')

In [56]:
index = TextSearch(
    text_fields=['section', 'question', 'text']
)
index.fit(documents)

index.search(
    query='I just signed up. Is it too late to join the course?',
    n_results=5,
    boost={'question': 3.0},
    filters={'course': 'data-engineering-zoomcamp'}
)

[{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
  'section': 'General course-related questions',
  'question': 'Course - Can I still join the course after the start date?',
  'course': 'data-engineering-zoomcamp'},
 {'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
  'section': 'General course-related questions',
  'question': 'Course - When will the course start?',
  'course': 'data-engineerin

**Note**: this is a toy example illustrating how relevance search works. It is not
meant for production — it scores every document on every query. Section 6 covers
what real engines do instead.

The same implementation, packaged:

https://github.com/alexeygrigorev/minsearch

## 4. Embeddings and Vector Search

Problem with text - only exact matches. How about synonyms?
What are Embeddings?

* Conversion to Numbers: Embeddings transform different words, sentences and documents into dense vectors (arrays with numbers).
* Capturing Similarity: They ensure similar items have similar numerical vectors, illustrating their closeness in terms of characteristics.
* Dimensionality Reduction: Embeddings reduce complex characteristics into vectors.
* Use in Machine Learning: These numerical vectors are used in machine learning models for tasks such as recommendations, text analysis, and pattern recognition.


### SVD

Singular Value Decomposition is methond of dimensionality reduction and is the simplest way to turn Bag-of-Words representation into embeddings

This way we still don't preserve the word order (because it wasn't in the Bag-of-Words representation) but we reduce dimensionality and capture synonyms.

We won't go into mathematics, it's sufficient to know that SVD "compresses" our input vectors in such a way that as much as possible of the original information is retained.

This compression is lossy compression - meaning that we won't be able to restore the 100% of the original vector, but the result is close enough.

http://wordvec.colorado.edu/papers/Deerwester_1990.pdf

In [57]:
from sklearn.decomposition import TruncatedSVD

X = matrices['text']
cv = vectorizers['text']

In [58]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 23808 stored elements and shape (948, 1333)>

`n_components=16` is the size of the embedding: every document goes from a
vocabulary-sized sparse row (1333 columns) down to 16 numbers.

In [59]:
svd = TruncatedSVD(n_components=16)
X_emb = svd.fit_transform(X)

In [60]:
X_emb.shape # from 1333 to 16

(948, 16)

In [61]:
X_emb[0] # this is called embedding,SVD try to capture from the original as possible,
# and similar words will be grouped, vectores capture similiarty between words. And this for documents

array([ 0.09652963, -0.08219315, -0.10296562, -0.07849389,  0.06821267,
       -0.06288122,  0.03600821, -0.05154157, -0.24204   , -0.29235417,
        0.0918795 , -0.10993662, -0.06366752, -0.13330534, -0.00336763,
       -0.00766128])

The query goes through exactly the same two steps — vectorizer, then SVD — so that
it lands in the same 16-dimensional space as the documents.

In [62]:
query = 'I just signed up. Is it too late to join the course?'

Q = cv.transform([query])
Q_emb = svd.transform(Q)
Q_emb[0]# represent the query

array([ 0.05790151, -0.03851679, -0.05668948, -0.027534  ,  0.0400553 ,
       -0.06340285,  0.01940065, -0.02814627, -0.16571148, -0.18777836,
        0.07420849, -0.09318738, -0.04576532, -0.08045028,  0.00666593,
       -0.01285773])

https://youtu.be/nMrGK5QgPVE?t=4321

In [63]:
np.dot(X_emb[0], Q_emb[0])

np.float64(0.15140557315133873)

In [64]:
score = cosine_similarity(X_emb, Q_emb).flatten()
score

array([ 9.87950400e-01,  8.68785057e-02,  9.81683805e-01,  9.27449926e-01,
       -2.85340381e-02,  6.56618120e-01,  6.53088661e-01,  9.87792223e-01,
        9.28181332e-01,  5.75989546e-01,  7.10872202e-01,  9.85837631e-01,
        9.13898554e-01,  9.72740370e-01,  1.53452918e-01,  9.80441844e-01,
        4.84360205e-01,  8.12861960e-01,  7.55140216e-01,  4.18921168e-01,
        5.76162820e-01, -3.97577944e-02,  8.65011413e-01,  7.05028107e-01,
        1.57336624e-01,  5.53978474e-02,  2.30531376e-01,  8.59849310e-01,
        6.16822546e-01,  9.12469239e-01,  3.92076306e-01,  2.43872799e-01,
        7.53861800e-01,  5.37445736e-01,  1.07212069e-01,  9.23712799e-01,
        4.53301015e-01,  2.05720623e-01,  6.68324715e-01,  5.95150203e-01,
        6.91923840e-01,  1.85355931e-01,  5.09852670e-01,  1.60841325e-01,
        7.34443894e-02,  1.22470421e-03, -1.34418220e-01, -1.66162920e-02,
        2.49239152e-02,  1.34887964e-01,  2.59733631e-01, -1.30909064e-01,
        3.96205719e-01,  

In [65]:
idx = np.argsort(-score)[:10]

In [66]:
list(df.loc[idx].text)

['Yes, you can. You won’t be able to submit some of the homeworks, but you can still take part in the course.\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers’ Projects by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
 "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue 

### Non-Negative Matrix Factorization

SVD creates values with negative numbers. It's difficult to interpet them. 

NMF (Non-Negative Matrix Factorization) is a similar concept, except for non-negative input matrices it produces non-negative results.

We can interpret each of the columns (features) of the embeddings as different topic/concepts and to what extent this document is about this concept.

Let's use it for the documents:

In [67]:
from sklearn.decomposition import NMF
nmf = NMF(n_components=16)
X_emb = nmf.fit_transform(X)
X_emb[0]

array([0.12805414, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.0008265 , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        ])

In [68]:
Q = cv.transform([query])
Q_emb = nmf.transform(Q)
Q_emb[0]

array([0.08530337, 0.00240387, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.0017905 ,
       0.        ])

In [79]:
score = cosine_similarity(X_emb, Q_emb).flatten()
idx = np.argsort(-score)[:10]
list(df.loc[idx].text)

['Please choose the closest one to your answer. Also do not post your answer in the course slack channel.',
 'Yes, you can. You won’t be able to submit some of the homeworks, but you can still take part in the course.\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers’ Projects by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
 "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
 "No, you can only get a certificate if you finish the course with a “live” cohort. We don't award certificates for the self-paced mode. The reason is you need to peer-review capstone(s) after submitting a project. You can only peer-review projects at the time the course is running.",
 "The purpose

### BERT 

The problem with the previous two approaches is that they don't take into account the word order. They just treat all the words separately (that's why it's called "Bag-of-Words")

BERT and other transformer models don't have this problem.

Let's create embeddings with BERT. We will use the Hugging Face library for that

In [80]:
!pip install transformers tqdm torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 67.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 41.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 69.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 65.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 24.2 MB/s  0:00:11 eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 35.6 MB/s  0:00:07 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 59.1 MB/s  0:00:02 eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 43.9 MB/s  0:00:04 eta 0:00:010:016
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 63.1 MB/s  0:00:00s eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 55.5 MB/s  0:00:03 eta 0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 62.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.

Use it:

In [69]:
import torch
from transformers import BertModel, BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
model.eval()  # Set the model to evaluation mode if not training

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

We need:

- tokenizer - for turning text into vectors
- model - for compressing the text into embeddings

First, we tokenize the text

In [70]:
texts = [
    "Yes, we will keep all the materials after the course finishes.",
    "You can follow the course at your own pace after it finishes"
]
encoded_input = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')

Then we compute the embeddings

In [71]:
with torch.no_grad():  # Disable gradient calculation for inference
    outputs = model(**encoded_input)
    hidden_states = outputs.last_hidden_state

`padding=True` pads every sequence in the batch to the same length, and
`truncation=True` cuts anything longer than the model's maximum input length.

The result has shape `(sentences, tokens, 768)` — one vector *per token*, not per
sentence.

In [72]:
hidden_states.shape

torch.Size([2, 15, 768])

Now we need to compress the embeddings:

In [73]:
sentence_embeddings = hidden_states.mean(dim=1)
sentence_embeddings.shape

torch.Size([2, 768])

And convert them to a numpy array

In [74]:
X_emb = sentence_embeddings.numpy()

Note that if you use a GPU, first you need to move your tensors to CPU

In [75]:
sentence_embeddings_cpu = sentence_embeddings.cpu()

Let's now compute it for our texts. We'll do it in batches. First, we define a function for batching:

In [76]:
def make_batches(seq, n):
    result = []
    for i in range(0, len(seq), n):
        batch = seq[i:i+n]
        result.append(batch)
    return result

And use it:

BERT is slow and memory-hungry compared to TF-IDF, and the whole corpus will not
fit in memory as one tensor, so we feed it in batches and stack the results.

In [77]:
from tqdm.auto import tqdm
texts = df['text'].tolist()
text_batches = make_batches(texts, 8)

all_embeddings = []

for batch in tqdm(text_batches):
    encoded_input = tokenizer(batch, padding=True, truncation=True, return_tensors='pt')

    with torch.no_grad():
        outputs = model(**encoded_input)
        hidden_states = outputs.last_hidden_state
        
        batch_embeddings = hidden_states.mean(dim=1)
        batch_embeddings_np = batch_embeddings.cpu().numpy()
        all_embeddings.append(batch_embeddings_np)

final_embeddings = np.vstack(all_embeddings)

  0%|          | 0/119 [00:00<?, ?it/s]

Let's put it into a function:

In [78]:
def compute_embeddings(texts, batch_size=8):
    text_batches = make_batches(texts, 8)
    
    all_embeddings = []
    
    for batch in tqdm(text_batches):
        encoded_input = tokenizer(batch, padding=True, truncation=True, return_tensors='pt')
    
        with torch.no_grad():
            outputs = model(**encoded_input)
            hidden_states = outputs.last_hidden_state
            
            batch_embeddings = hidden_states.mean(dim=1)
            batch_embeddings_np = batch_embeddings.cpu().numpy()
            all_embeddings.append(batch_embeddings_np)
    
    final_embeddings = np.vstack(all_embeddings)
    return final_embeddings

And use it:

In [79]:
embeddings = {}

In [80]:
# fields = ['section', 'question', 'text']

for f in fields:
    print(f'computing embeddings for {f}...')
    embeddings[f] = compute_embeddings(df[f].tolist())

computing embeddings for section...


  0%|          | 0/119 [00:00<?, ?it/s]

computing embeddings for question...


  0%|          | 0/119 [00:00<?, ?it/s]

computing embeddings for text...


  0%|          | 0/119 [00:00<?, ?it/s]

## 5. Combining Text and Vector Search

The two methods fail in opposite ways:

* keyword / TF-IDF search is precise on exact words and blind to synonyms
* vector search catches synonyms and paraphrases, but happily returns something
  vaguely on-topic that misses the exact term you typed

So run both and merge. The simplest merge is a weighted sum of the two scores —
one number, `alpha`, decides how much we trust each side.

In [81]:
query = 'I just signed up. Is it too late to join the course?'

q = vectorizers['text'].transform([query])   # sparse, vocabulary space
q_svd = svd.transform(q)                     # dense, 16-dim space
X_svd = svd.transform(matrices['text'])

text_score = cosine_similarity(matrices['text'], q).flatten()
vector_score = cosine_similarity(X_svd, q_svd).flatten()

alpha = 0.5
score = alpha * text_score + (1 - alpha) * vector_score

idx = np.argsort(-score)[:5]
list(df.iloc[idx].text)

["The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'Yes, you can. You won’t be able to submit some of the homeworks, but you can still take part in the course.\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers’ Projects by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
 "The process is automated now, so you should receive the email eventually. If you haven’t, check your promotions tab in Gmail as well as spam.\nIf you u

In [82]:
# compare what each side finds on its own
print('text only  :', np.argsort(-text_score)[:5])
print('vector only:', np.argsort(-vector_score)[:5])
print('combined   :', np.argsort(-score)[:5])

text only  : [  0 440 449 448  22]
vector only: [449   0   7 764 436]
combined   : [  0 449 440 764 452]


`alpha = 1.0` is pure keyword search, `alpha = 0.0` is pure vector search.

To use BERT instead of SVD here, swap `X_svd` for `embeddings['text']` and
`q_svd` for `compute_embeddings([query])`.

**Caveat.** A weighted sum only makes sense if both scores are on the same scale.
Here they are — both are cosine similarities — but a raw BM25 score and a cosine
similarity are not comparable, and adding them lets whichever has the bigger
numbers dominate.

The scale-free alternative is to combine *ranks* instead of scores. Reciprocal
Rank Fusion (RRF) gives each document `1 / (k + rank)` from each ranking and adds
those up.

In [83]:
from collections import defaultdict

def rrf(*rankings, k=60):
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] += 1 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

text_rank = np.argsort(-text_score)[:10]
vector_rank = np.argsort(-vector_score)[:10]

idx = rrf(text_rank, vector_rank)[:5]
list(df.iloc[idx].text)

["The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'Yes, you can. You won’t be able to submit some of the homeworks, but you can still take part in the course.\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers’ Projects by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
 'If you have submitted two projects (and peer-reviewed at least 3 course-mates’ projects for each submission), you will get the certificate for the cour

Documents that both methods like float to the top; documents only one method found
still get in, just lower. The constant `k` damps the influence of the very top
positions — it is a knob, and 60 is the value from the original paper.

## 6. Practical Implementation Aspects and Tools

So far every search we ran compared the query to **every single document**.

With 1000 FAQ entries that is fine, it finishes instantly. With 100 million
documents it never finishes.

Real search engines don't do it. They first throw away almost everything, then
score only the handful of documents left. There is one trick for text and one
trick for vectors. Both are below.

### Inverted indexes for text search

Think of the index at the back of a textbook. To find "kafka" you don't read all
800 pages, you look the word up and it hands you the page numbers.

That is exactly an inverted index:

* forward (what we had): document 5 &rarr; the words inside it
* inverted: the word "kafka" &rarr; the documents that contain it

Build it once. Then a query only has to look at documents that share at least one
word with it.

In [84]:
import re
from collections import defaultdict

def tokenize(text):
    return re.findall(r'[a-z0-9]+', text.lower())

inverted_index = defaultdict(set)

for doc_id, text in enumerate(df.text):
    for token in tokenize(text):
        inverted_index[token].add(doc_id)

len(inverted_index)   # vocabulary size

6130

In [85]:
# look-ups are just set operations
print('both  :', len(inverted_index['docker'] & inverted_index['kafka']))
print('either:', len(inverted_index['docker'] | inverted_index['kafka']))

both  : 3
either: 157


In [86]:
def candidates(query):
    ids = set()
    for token in tokenize(query):
        ids |= inverted_index.get(token, set())
    return sorted(ids)

c = candidates('how do I run kafka in docker')
len(c), len(df)   # score this many documents instead of all of them

(709, 948)

`candidates()` returned a short list. TF-IDF (or BM25) now scores only that list
instead of the whole dataset. Same results, a fraction of the work.

Real engines also store *how often* each word appears next to each document id, so
the scoring step never has to open the original text again.

### LSH for vector search (random projections)

Vectors have no words, so there is nothing to look up. We need a different trick:
**drop similar vectors into the same bucket, then search only one bucket.**

A normal hash (MD5, SHA) works hard to give two similar inputs completely
different outputs. Locality-Sensitive Hashing wants the opposite: similar inputs
*should* collide.

The easiest one for cosine similarity is random projections:

1. draw a few random lines through the middle of the space
2. for each line ask: is this vector on the left or the right? &rarr; `1` or `0`
3. glue the bits together, that string is the bucket name

Two vectors pointing in the same direction land on the same side of most lines, so
they get the same string, so they share a bucket.

In [87]:
np.random.seed(1)

n_bits = 6
planes = np.random.randn(X_svd.shape[1], n_bits)

def lsh_hash(V):
    bits = (V @ planes) > 0
    return [''.join('1' if b else '0' for b in row) for row in bits]

buckets = defaultdict(list)
for doc_id, h in enumerate(lsh_hash(X_svd)):
    buckets[h].append(doc_id)

len(buckets), sorted(map(len, buckets.values()), reverse=True)[:5]

(51, [198, 71, 48, 43, 43])

In [88]:
# at query time we hash the query and only score its bucket
bucket = buckets.get(lsh_hash(q_svd)[0], [])
len(bucket), len(df)   # if this comes back empty, lower n_bits

(43, 948)

In [89]:
bucket_score = cosine_similarity(X_svd[bucket], q_svd).flatten()
idx = np.array(bucket)[np.argsort(-bucket_score)[:5]]
list(df.iloc[idx].text)

['Yes, you can. You won’t be able to submit some of the homeworks, but you can still take part in the course.\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers’ Projects by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
 "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue 

This is **approximate**, and that is the whole point. A document sitting just on
the wrong side of one line falls into another bucket and we never look at it. We
lose a few good results and gain a lot of speed.

One knob, two directions:

* more bits &rarr; more, smaller buckets &rarr; faster, but more misses
* fewer bits &rarr; fewer, bigger buckets &rarr; slower, but safer

Real implementations build several tables, each with its own random lines, and
search every matching bucket. A miss in one table is usually caught by another.

HNSW, the index most vector databases use today, does the same job with a graph
instead of buckets. Different structure, identical deal: skip almost everything,
accept a few misses.

### Technologies

Nobody writes the above by hand in production. Here is what people actually use.

**Text search**

| Tool | What it is |
|---|---|
| **Lucene** | The Java library that does the inverted index and BM25 scoring. The actual engine. |
| **Elasticsearch** / **OpenSearch** | Lucene wrapped in a server: HTTP API, many machines, replication. OpenSearch is the open-source fork of Elasticsearch. |

**Vector search**

| Tool | What it is |
|---|---|
| **FAISS** | A library of fast approximate-nearest-neighbour indexes. Runs inside your Python process, no server, no storage. |
| **Qdrant, Weaviate, Milvus, Chroma, Pinecone** | Vector *databases*: the same indexes plus storage, filtering on metadata, and an API. |
| **pgvector** | A Postgres extension. Use it when you'd rather not run one more service. |

Good news for section 5: Elasticsearch, OpenSearch, Qdrant and most of the others
already do hybrid search for you. Send both queries, they fuse the results with
RRF.

## 7. Conclusion — the road from keyword filtering to a fine-tuned LLM

Every step in this notebook exists because the step before it failed at something
specific. Read the diagram top to bottom: the left column is the idea, the arrow
label is the problem that forced the next idea, the right branches are the
production tools that shipped each idea.

```
  ══════════ 1. EXACT WORDS ═══════════════════════════════════════════════

   keyword filtering  (boolean retrieval, 1960s)
          │  matches or doesn't - no way to rank 900 hits
          ▼
   TF-IDF  (Sparck Jones, 1972)
          │  now we have a weight per word, but no way to compare two docs
          ▼
   cosine similarity / vector space model  (Salton, 1975)
          │  long documents still game the score
          ▼
   BM25  (Robertson & Walker, 1994) ────► Lucene 1999 ──► Elasticsearch 2010
          │                                                       │
          │                                                       ▼
          │                                                  OpenSearch 2021
          │  "sign up" still never matches "register"
          ▼
  ══════════ 2. MEANING, BAG OF WORDS ═════════════════════════════════════

   LSA / SVD  (Deerwester et al., 1990)
          │  16 dense numbers instead of 1333 sparse ones - synonyms collapse
          │  together. But the numbers go negative and mean nothing to a human
          ▼
   NMF  (Lee & Seung, 1999)
          │  non-negative, so each column reads as a topic
          │  still no word order: "docker in kafka" == "kafka in docker"
          ▼
  ══════════ 3. MEANING, WORD ORDER ═══════════════════════════════════════

   word2vec / GloVe  (2013 / 2014)
          │  vectors learned from context, not counts - but one vector per word,
          │  so "bank" has a single meaning
          ▼
   Transformer  (Vaswani et al., 2017)
          │  attention: every token is read in the context of the others
          ▼
   BERT  (Devlin et al., 2018)
          │  contextual embeddings - but mean-pooled BERT (what we did above)
          │  was never trained to make similar sentences land close together
          ▼
   Sentence-BERT 2019 / DPR 2020 ──► LSH 1998 ──► HNSW 2016, FAISS 2017
          │   embeddings trained                          │
          │   for retrieval                               ▼
          │                                    vector DBs: Qdrant, Weaviate,
          │  vectors miss the exact term       Milvus, Chroma, pgvector
          │  you actually typed
          ▼
  ══════════ 4. BOTH AT ONCE ══════════════════════════════════════════════

   hybrid search: BM25 + dense vectors, fused with RRF  (Cormack et al., 2009)
          │  we now return excellent documents. The user wanted an answer
          ▼
  ══════════ 5. ANSWERS, NOT LINKS ════════════════════════════════════════

   RAG  (Lewis et al., 2020)  =  this search engine  +  an LLM
          │  retrieval grounds the model in your documents instead of its weights
          ▼
   LLM  (GPT-3, 2020)  ──►  instruction tuning / RLHF  (InstructGPT, 2022)
          │  generic model, generic tone, no knowledge of your domain jargon
          ▼
   fine-tuning  (full FT, or LoRA / PEFT, Hu et al., 2021)
             teach format, tone and domain vocabulary - NOT facts:
             facts stay in the retrieval layer, because they change
```

### Why each step exists

| # | Step | The problem it solves | Why it could not come earlier |
|---|------|----------------------|-------------------------------|
| 1 | Keyword filtering | Find documents containing a term at all | The baseline |
| 2 | TF-IDF | 900 documents match — which one first? Rare words carry more signal than common ones | Needs a corpus to compute document frequency |
| 3 | Cosine similarity | Turn two weight vectors into one comparable number, independent of length | Needs vectors, i.e. the vector space model |
| 4 | BM25 | The 20th "kafka" in a doc is not worth as much as the 2nd; long docs shouldn't win by default | A probabilistic refinement of TF-IDF |
| 5 | Inverted index → Lucene → Elasticsearch/OpenSearch | Scoring all N documents per query doesn't scale | Engineering, not new maths: needed the scoring formula to be settled first |
| 6 | SVD / LSA | Synonyms. "sign up" and "register" are different columns and can never match | Needs the term-document matrix from steps 2–3 |
| 7 | NMF | SVD components go negative and are uninterpretable | Same input, a constraint added |
| 8 | word2vec / GloVe | LSA meaning comes from co-occurrence counts in one corpus; learn it from context on billions of words instead | Needed cheap GPUs and large text dumps |
| 9 | Transformer | Bag of words and static word vectors both throw away order and context | Needed attention (2014) plus the hardware to train it |
| 10 | BERT | One vector per word can't disambiguate; BERT gives one vector *per occurrence* | Built directly on the Transformer encoder |
| 11 | Sentence-BERT / DPR | Mean-pooled BERT is a weak similarity metric — it was trained to fill in masked words, not to rank | Needs labelled query–document pairs to fine-tune on |
| 12 | LSH / HNSW / FAISS / vector DBs | Comparing a query to 100M dense vectors is a full scan | Only becomes a problem once dense retrieval actually works |
| 13 | Hybrid search + RRF | Lexical misses synonyms, dense misses exact terms, IDs and error codes | Needs both retrievers to exist |
| 14 | RAG | The LLM's knowledge is frozen at training time and it invents citations | Needs a retriever *and* a model good enough to read the passages |
| 15 | LLM + instruction tuning | Users want an answer, not ten links | Needs the scale of GPT-3 and then RLHF to follow instructions |
| 16 | Fine-tuning / LoRA | Format, tone, domain vocabulary the base model gets wrong | Last resort: it's the expensive knob, and retrieval already fixed the facts |

### The practical takeaway

The order above is also the order to **build** in. Steps 1–5 (BM25 in Elasticsearch)
solve most of a real search problem for a fraction of the cost. Add dense retrieval
when you measure that synonyms are actually hurting recall, hybrid when you measure
that dense alone drops exact matches, and fine-tune last — after retrieval, prompting
and the ranking are all exhausted.

This notebook stopped at step 13. Step 14 onwards is the
[LLM RAG workshop](https://github.com/alexeygrigorev/llm-rag-workshop).